# Splitting documents

This notebook showcases the document splitting process, and looking up chunks using the embedding model.

In [ ]:
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

from shared.utils.model import get_embedding_model

In [ ]:
reader = PdfReader("data/Attention Is All You Need.pdf")
document = "\n".join(page.extract_text() for page in reader.pages)

In [ ]:
print(f"Document length: {len(document)} chars")

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(document)

print(f"Chunked {len(document)} chars into {len(chunks)} chunks")

In [ ]:
chroma = chromadb.Client()

collection = chroma.get_or_create_collection("embeddings")

In [ ]:
model = get_embedding_model()

In [ ]:
embedding = model.encode(chunks, show_progress_bar=True, batch_size=1)

In [ ]:
collection.count()

collection.add(
    ids=[f"test-{i}" for i in range(len(chunks))],
    embeddings=embedding,
    documents=chunks
)

In [ ]:
qe = model.encode("transformer design implementation")

collection.query(query_embeddings=qe, n_results=1)

In [ ]:
collection.get(include=["embeddings"])